# **Qwen-Image-2.1 for Image Generation & Editing (ComfyUI)**
- Run the cell below to install dependencies, download models, and get a link (e.g. https://literature-consortium-align.trycloudflare.com) which you can use to launch the ComfyUI interface.
- **Workflows**: https://github.com/Comfy-Org/workflow_templates/blob/main/templates/
- **github page**: https://github.com/QwenLM/Qwen-Image-2.1

- **Notebook source**: https://github.com/Isi-dev/Google-Colab_Notebooks

In [ ]:
# @title # 1. 💥 Prepare Environment & Install Dependencies {"single-column":true}
import os

# 1. ALWAYS force the directory back to the Colab root before cloning
%cd /content

# 2. Clone ComfyUI only if it doesn't already exist in the root
if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

# 3. Move into the correct, top-level ComfyUI folder
%cd /content/ComfyUI

# Install core dependencies
!pip install -r requirements.txt
from IPython.display import clear_output

# 4. Install custom nodes specifically for Qwen-Image 2.1 GGUF support
if not os.path.exists("custom_nodes/ComfyUI-GGUF"):
    !cd custom_nodes && git clone https://github.com/leejet/ComfyUI-GGUF.git

# Install GGUF and high-speed Hugging Face download engine
!pip install gguf huggingface_hub hf_transfer

clear_output()

# @markdown Select your desired models below.

# @markdown ---
# @markdown ### **HuggingFace Token (Optional but Recommended)**
# @markdown Adding a free token prevents rate-limiting and stalling on massive downloads. Get yours at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
hf_token = "" # @param {type:"string"}

# @markdown ---
# @markdown ### **Text Encoders**
text_encoder_model = "qwen3vl_8b_int8_convrot.safetensors" # @param ["qwen3vl_8b_bf16.safetensors", "qwen3vl_8b_int8_convrot.safetensors", "qwen3vl_8b_w4a8.safetensors", "None"]
download_custom_text_encoder = False # @param {type:"boolean"}
custom_text_encoder_url = "" # @param {type:"string"}

# @markdown ---
# @markdown ### **Prompt Enhancers (Optional)**
# @markdown Fine-tuned LLMs that expand basic prompts into detailed templates for Qwen-Image 2.1.
download_prompt_enhancer_t2i = False # @param {type:"boolean"}
download_prompt_enhancer_i2i = False # @param {type:"boolean"}
download_custom_prompt_enhancer = False # @param {type:"boolean"}
custom_prompt_enhancer_url = "" # @param {type:"string"}

download_vae = True

# @markdown ---
# @markdown ### **UNet / Main Models**
unet_model = "qwen_image_2.1_int8_convrot.safetensors (Comfy-Org)" # @param ["qwen_image_2.1_bf16.safetensors (Comfy-Org)", "qwen_image_2.1_int8_convrot.safetensors (Comfy-Org)", "qwen-image-2.1-Q4_0.gguf (abenzerps)", "qwen-image-2.1-Q4_K_M.gguf (abenzerps)", "qwen-image-2.1-Q5_K_M.gguf (abenzerps)", "qwen-image-2.1-Q6_K.gguf (abenzerps)", "qwen-image-2.1-Q8_0.gguf (abenzerps)", "qwen-image-2.1-Q4_0.gguf (Uncensored)", "qwen-image-2.1-Q4_K_M.gguf (Uncensored)", "qwen-image-2.1-Q5_K_M.gguf (Uncensored)", "qwen-image-2.1-Q6_K.gguf (Uncensored)", "qwen-image-2.1-Q8_0.gguf (Uncensored)", "None"]
download_custom_unet_model = False # @param {type:"boolean"}
custom_unet_model_url = "" # @param {type:"string"}

# @markdown ---
# @markdown ### **Custom LoRAs (Optional)**
# @markdown Check the box to download the corresponding LoRA URL. You can paste any direct download link here.
download_lora_1 = False # @param {type:"boolean"}
lora_url_1 = "" # @param {type:"string"}

download_lora_2 = False # @param {type:"boolean"}
lora_url_2 = "" # @param {type:"string"}

download_lora_3 = False # @param {type:"boolean"}
lora_url_3 = "" # @param {type:"string"}

download_lora_4 = False # @param {type:"boolean"}
lora_url_4 = "" # @param {type:"string"}

download_lora_5 = False # @param {type:"boolean"}
lora_url_5 = "" # @param {type:"string"}

import os
import subprocess
from huggingface_hub import hf_hub_download

%cd /content/ComfyUI
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

def download_file(repo_id, repo_relative_path, dest_dir, token):
    if not repo_relative_path or "None" in repo_relative_path:
        return

    os.makedirs(dest_dir, exist_ok=True)
    filename_only = os.path.basename(repo_relative_path)
    destination_file = os.path.join(dest_dir, filename_only)

    if os.path.exists(destination_file):
        print(f"⏭️ Skipping {filename_only} (Already exists)\n")
        return

    print(f"Downloading {filename_only} from {repo_id}...")
    try:
        hf_hub_download(
            repo_id=repo_id,
            filename=repo_relative_path,
            local_dir=dest_dir,
            local_dir_use_symlinks=False,
            token=token if token else None
        )
        print(f"✅ Successfully downloaded {filename_only}\n")
    except Exception as e:
        print(f"❌ Error downloading {filename_only}: {str(e)}\n")

def download_custom_url(url, dest_folder, token=""):
    if not url or not str(url).strip():
        return
    os.makedirs(dest_folder, exist_ok=True)
    filename = url.split('/')[-1].split('?')[0] or "downloaded_model"
    dest_path = os.path.join(dest_folder, filename)
    if os.path.exists(dest_path):
        return
    print(f"Downloading {filename}...")
    headers = ["--header", f"Authorization: Bearer {token}"] if token and "huggingface.co" in url else []
    subprocess.run(["wget", "-q", "--show-progress", "-c", url, "-O", dest_path] + headers, check=False)

download_queue = []
token_str = hf_token.strip()

# 1. Text Encoders
if download_custom_text_encoder and custom_text_encoder_url.strip():
    download_custom_url(custom_text_encoder_url.strip(), "models/text_encoders", token_str)
elif text_encoder_model != "None":
    download_queue.append(("Comfy-Org/Qwen-Image-2.1", f"text_encoders/{text_encoder_model}", "models/text_encoders"))

# Handle Prompt Enhancers
if download_prompt_enhancer_t2i:
    download_queue.append((
        "Comfy-Org/Qwen-Image-2.1",
        "text_encoders/qwen3.5_9b_qwen_image_2.1_pe_t2i.int8_convrot.safetensors",
        "models/text_encoders"
    ))

if download_prompt_enhancer_i2i:
    download_queue.append((
        "Comfy-Org/Qwen-Image-2.1",
        "text_encoders/qwen3.5_9b_qwen_image_2.1_pe_i2i.int8_convrot.safetensors",
        "models/text_encoders"
    ))

if download_custom_prompt_enhancer and custom_prompt_enhancer_url.strip():
    download_custom_url(custom_prompt_enhancer_url.strip(), "models/text_encoders", token_str)

# 2. VAE
if download_vae:
    download_queue.append(("Comfy-Org/Qwen-Image-2.1", "vae/qwen_image_2.1_vae_bf16.safetensors", "models/vae"))

# 3. UNet Models Mapping
unet_mapping = {
    "qwen_image_2.1_bf16.safetensors (Comfy-Org)": ("Comfy-Org/Qwen-Image-2.1", "diffusion_models/qwen_image_2.1_bf16.safetensors"),
    "qwen_image_2.1_int8_convrot.safetensors (Comfy-Org)": ("Comfy-Org/Qwen-Image-2.1", "diffusion_models/qwen_image_2.1_int8_convrot.safetensors"),
    "qwen-image-2.1-Q4_0.gguf (abenzerps)": ("abenzerps/Qwen-Image-2.1-GGUF", "qwen-image-2.1-Q4_0.gguf"),
    "qwen-image-2.1-Q4_K_M.gguf (abenzerps)": ("abenzerps/Qwen-Image-2.1-GGUF", "qwen-image-2.1-Q4_K_M.gguf"),
    "qwen-image-2.1-Q5_K_M.gguf (abenzerps)": ("abenzerps/Qwen-Image-2.1-GGUF", "qwen-image-2.1-Q5_K_M.gguf"),
    "qwen-image-2.1-Q6_K.gguf (abenzerps)": ("abenzerps/Qwen-Image-2.1-GGUF", "qwen-image-2.1-Q6_K.gguf"),
    "qwen-image-2.1-Q8_0.gguf (abenzerps)": ("abenzerps/Qwen-Image-2.1-GGUF", "qwen-image-2.1-Q8_0.gguf"),
    "qwen-image-2.1-Q4_0.gguf (Uncensored)": ("KasugaiSakura/Qwen-Image-2.1-Uncensored-GGUF", "qwen-image-2.1-Q4_0.gguf"),
    "qwen-image-2.1-Q4_K_M.gguf (Uncensored)": ("KasugaiSakura/Qwen-Image-2.1-Uncensored-GGUF", "qwen-image-2.1-Q4_K_M.gguf"),
    "qwen-image-2.1-Q5_K_M.gguf (Uncensored)": ("KasugaiSakura/Qwen-Image-2.1-Uncensored-GGUF", "qwen-image-2.1-Q5_K_M.gguf"),
    "qwen-image-2.1-Q6_K.gguf (Uncensored)": ("KasugaiSakura/Qwen-Image-2.1-Uncensored-GGUF", "qwen-image-2.1-Q6_K.gguf"),
    "qwen-image-2.1-Q8_0.gguf (Uncensored)": ("KasugaiSakura/Qwen-Image-2.1-Uncensored-GGUF", "qwen-image-2.1-Q8_0.gguf")
}

if download_custom_unet_model and custom_unet_model_url.strip():
    download_custom_url(custom_unet_model_url.strip(), "models/diffusion_models", token_str)
elif unet_model != "None":
    repo, filename = unet_mapping[unet_model]
    download_queue.append((repo, filename, "models/diffusion_models"))

for repo, rel_path, dest_dir in download_queue:
    download_file(repo, rel_path, dest_dir, token_str)


# Execute LoRA URL downloads
print("Initializing Custom LoRA Downloads...\n")
loras_to_download = [
    (download_lora_1, lora_url_1),
    (download_lora_2, lora_url_2),
    (download_lora_3, lora_url_3),
    (download_lora_4, lora_url_4),
    (download_lora_5, lora_url_5),
]

for should_download, lora_url in loras_to_download:
    if should_download and lora_url.strip():
        download_custom_url(lora_url.strip(), "models/loras", token_str)

clear_output()

# @title 3. 🚀 Run ComfyUI
use_cloudflare = True # @param {type:"boolean"}
use_interface_in_cell = False # @param {type:"boolean"}
vram_management = "Normal VRAM (Load/Unload)" # @param ["Normal VRAM (Load/Unload)", "High VRAM (Keep loaded)"]

import torch
import os
from IPython.display import clear_output
clear_output()
%cd /content/ComfyUI

launch_args = "--enable-cors-header"
if vram_management == "High VRAM (Keep loaded)":
    launch_args += " --highvram"

if use_cloudflare:
    if not os.path.exists("cloudflared-linux-amd64.deb"):
        !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
        !dpkg -i cloudflared-linux-amd64.deb

    import subprocess
    import threading
    import time
    import socket

    def iframe_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            result = sock.connect_ex(('127.0.0.1', port))
            if result == 0:
                break
            sock.close()
        print("\nComfyUI finished loading, launching Cloudflare tunnel...\n")
        p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        for line in p.stderr:
            l = line.decode()
            if "trycloudflare.com " in l:
                print("This is your ComfyUI URL:", l[l.find("http"):], end='')
        clear_output()

    threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
    !python main.py $launch_args

elif use_interface_in_cell:
    import threading, time, socket
    def iframe_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            if sock.connect_ex(('127.0.0.1', port)) == 0:
                break
            sock.close()
        from google.colab import output
        output.serve_kernel_port_as_iframe(port, height=1024)
        clear_output()
        print("To open in a standalone window click here:")
        output.serve_kernel_port_as_window(port)
    threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
    !python main.py $launch_args

else:
    import socket, time, threading
    from google.colab import output
    def link_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            if sock.connect_ex(('127.0.0.1', port)) == 0:
                break
            sock.close()
        clear_output()
        print("Click the link below to launch the ComfyUI interface:")
        output.serve_kernel_port_as_window(port)
    threading.Thread(target=link_thread, daemon=True, args=(8188,)).start()
    !python main.py $launch_args